# Code mẫu — NumPy

NumPy tính một lúc trên rất nhiều phần tử, nhưng kết quả chỉ có nghĩa khi ta hiểu đúng shape, dtype và từng trục. Các ví dụ dưới đây dùng bảng nhỏ để kiểm được từng ô bằng tay, rồi mới ghép lại thành phép tính trên cả mảng.

## Ví dụ 1: Trung bình theo môn, theo học sinh, và theo trọng số

Ba học sinh có điểm hai môn, mỗi hàng là một học sinh: `[4, 6]`, `[6, 8]`, `[8, 10]`. Ta cần tính trung bình của từng môn, trung bình của từng học sinh, và điểm tổng hợp khi môn đầu chiếm 25%, môn sau chiếm 75%.

In [ ]:
import numpy as np

scores = np.array([[4, 6], [6, 8], [8, 10]], dtype=np.float64)
weights = np.array([0.25, 0.75])
print("shape:", scores.shape)
print("từng môn:", scores.mean(axis=0).tolist())
print("từng học sinh:", scores.mean(axis=1).tolist())

# Phép @ nhân hai điểm với hai trọng số rồi cộng lại, ra một điểm cho mỗi hàng.
weighted_scores = scores @ weights
print("có trọng số:", weighted_scores.tolist())
assert np.allclose(weighted_scores, (scores * weights).sum(axis=1))

row_mean = scores.mean(axis=1, keepdims=True)
print("shape trung bình hàng:", row_mean.shape)
print("lệch trung bình hàng:", (scores - row_mean).tolist())

shape: (3, 2)
từng môn: [6.0, 8.0]
từng học sinh: [5.0, 7.0, 9.0]
có trọng số: [5.5, 7.5, 9.5]
shape trung bình hàng: (3, 1)
lệch trung bình hàng: [[-1.0, 1.0], [-1.0, 1.0], [-1.0, 1.0]]


`axis=0` gộp ba học sinh lại, nên còn hai con số, mỗi môn một số. `axis=1` gộp hai môn lại, nên còn ba con số, mỗi học sinh một số. Học sinh đầu có điểm tổng hợp `4 × 0.25 + 6 × 0.75 = 5.5`, cao hơn trung bình thường là 5, vì môn có điểm cao hơn được nhân trọng số lớn hơn. `keepdims=True` giữ shape `(3, 1)`, nhờ vậy phép trừ ở dòng cuối trừ đúng trung bình của từng hàng.

## Ví dụ 2: Chuẩn hóa dữ liệu mới bằng thống kê đã tính trên train

Train có ba hàng `[1, 10]`, `[3, 10]`, `[5, 10]`. Test có hai hàng `[7, 10]`, `[9, 10]`. Cột đầu thay đổi, còn cột sau lúc nào cũng bằng 10. Ta muốn đo độ lệch của train và test bằng cùng một thước. Sau khi chuẩn hóa, các con số không còn đơn vị ban đầu nữa.

In [ ]:
import numpy as np

train = np.array([[1, 10], [3, 10], [5, 10]], dtype=np.float64)
test = np.array([[7, 10], [9, 10]], dtype=np.float64)

mean = train.mean(axis=0)
std = train.std(axis=0, ddof=0)
# Cột hằng có độ lệch chuẩn bằng 0. Thay mẫu số bằng 1 để không chia cho 0.
scale = np.where(std == 0, 1.0, std)
train_z = (train - mean) / scale
test_z = (test - mean) / scale
print("mean train:", mean.tolist())
print("std train:", std.round(3).tolist())
print("train đã đổi:", train_z.round(3).tolist())
print("test dùng thống kê train:", test_z.round(3).tolist())

# Làm sai để so: tính lại thống kê trên test là đổi luôn cả thước đo.
test_std = test.std(axis=0, ddof=0)
test_scale = np.where(test_std == 0, 1.0, test_std)
test_refitted = (test - test.mean(axis=0)) / test_scale
print("test học lại thống kê:", test_refitted.round(3).tolist())
assert np.allclose(train_z.mean(axis=0), 0)
assert np.isfinite(train_z).all() and np.isfinite(test_z).all()

mean train: [3.0, 10.0]
std train: [1.633, 0.0]
train đã đổi: [[-1.225, 0.0], [0.0, 0.0], [1.225, 0.0]]
test dùng thống kê train: [[2.449, 0.0], [3.674, 0.0]]
test học lại thống kê: [[-1.0, 0.0], [1.0, 0.0]]


Cột đầu của train có trung bình 3 và phương sai `(4 + 0 + 4) / 3 = 8/3`, nên độ lệch chuẩn khoảng 1.633. Giá trị 7 của test cao hơn trung bình train khoảng 2.449 lần độ lệch chuẩn, tức là một giá trị khá lớn so với những gì model đã thấy. Nếu tính lại thống kê trên test, cũng giá trị ấy lại thành −1, như thể nó nhỏ hơn mức trung bình. Hai con số trả lời hai câu hỏi khác nhau, và model chỉ hiểu câu hỏi thứ nhất. Cột hằng bằng 10 thành 0 ở cả hai tập, và không sinh ra `NaN`.

## Ví dụ 3: Tăng sáng ảnh mà không làm hỏng ảnh gốc

Một ảnh xám 2 × 2 được lưu bằng `uint8`, mỗi ô nằm trong khoảng 0 đến 255. Ta muốn sửa thử một phần ảnh cắt ra, và tăng sáng cả ảnh thêm 30. Có hai cái bẫy: phần cắt có thể dùng chung bộ nhớ với ảnh gốc, còn phép cộng trên `uint8` có thể bị tràn và quay vòng.

In [ ]:
import numpy as np

raw = np.array([[240, 10], [100, 200]], dtype=np.uint8)
crop = raw[:, :1]
crop[0, 0] = 0
print("sửa view, gốc đổi:", raw.tolist())
print("dùng chung bộ nhớ:", np.shares_memory(raw, crop))

raw = np.array([[240, 10], [100, 200]], dtype=np.uint8)
crop = raw[:, :1].copy()
crop[0, 0] = 0
print("sửa copy, giữ gốc:", raw.tolist())

wrapped = raw + 30
# Đổi sang kiểu rộng hơn trước khi cộng, rồi dùng clip để kẹp về khoảng 0–255.
bright = np.clip(raw.astype(np.int16) + 30, 0, 255).astype(np.uint8)
print("cộng uint8:", wrapped.tolist())
print("đổi kiểu rồi kẹp:", bright.tolist())
assert np.array_equal(raw, [[240, 10], [100, 200]])

sửa view, gốc đổi: [[0, 10], [100, 200]]
dùng chung bộ nhớ: True
sửa copy, giữ gốc: [[240, 10], [100, 200]]
cộng uint8: [[14, 40], [130, 230]]
đổi kiểu rồi kẹp: [[255, 40], [130, 230]]


Lần cắt đầu tiên làm ô 240 của ảnh gốc thành 0, vì phần cắt chỉ là một cửa sổ nhìn vào cùng vùng nhớ. Thêm `.copy()` thì ảnh gốc được giữ nguyên. Với phép cộng thẳng, `240 + 30 = 270` vượt quá 255 nên quay vòng thành `270 − 256 = 14`, tức một điểm rất sáng bỗng thành gần đen. Đổi kiểu trước rồi kẹp thì được 255, đúng ý tăng sáng. Kẹp sau khi đã tràn thì vô ích, vì con số 270 đã mất từ trước.

## Ví dụ 4: Gán nhãn theo điểm mẫu gần nhất, và bắt một lỗi shape không báo lỗi

Có ba điểm mẫu trên mặt phẳng: A ở `(0, 0)`, B ở `(3, 4)`, C ở `(6, 0)`. Ba điểm mới cần nhận nhãn của điểm mẫu gần nó nhất. Hai tọa độ cùng tính bằng mét. Để biết điểm nào gần nhất thì so bình phương khoảng cách là đủ, chưa cần lấy căn.

In [ ]:
import numpy as np

references = np.array([[0, 0], [3, 4], [6, 0]], dtype=np.float64)
labels = np.array(["A", "B", "C"])
queries = np.array([[0, 3], [4, 4], [5, 1]], dtype=np.float64)

# Ba trục lần lượt là: điểm mới, điểm mẫu, tọa độ.
delta = queries[:, None, :] - references[None, :, :]
distance2 = (delta ** 2).sum(axis=2)
nearest = distance2.argmin(axis=1)
predicted = labels[nearest]
print("shape trung gian:", delta.shape)
print("bình phương khoảng cách:", distance2.tolist())
print("nhãn dự đoán:", predicted.tolist())
print("byte của delta:", delta.nbytes)

# Tính lại từng cặp bằng vòng lặp thường để kiểm phép broadcast ở trên.
expected_distance2 = np.array([
    [sum((float(q[k]) - float(r[k])) ** 2 for k in range(2))
     for r in references] for q in queries
])
assert np.array_equal(distance2, expected_distance2)

shape trung gian: (3, 3, 2)
bình phương khoảng cách: [[9.0, 10.0, 45.0], [32.0, 1.0, 20.0], [26.0, 13.0, 2.0]]
nhãn dự đoán: ['A', 'B', 'C']
byte của delta: 144


Hàng đầu có các bình phương khoảng cách là 9, 10 và 45, nên điểm `(0, 3)` gần A nhất. Mảng trung gian có `3 × 3 × 2` số `float64`, tốn `18 × 8 = 144` byte. Khi số điểm tăng, mảng này lớn theo tích của số điểm mới và số điểm mẫu, nên với dữ liệu lớn phải chia thành từng phần. Nếu có hai khoảng cách bằng nhau, `argmin` chọn vị trí xuất hiện trước, tức thứ tự các điểm mẫu chính là luật phá hòa của đoạn code này.

Đáp án đúng ở đây là `A, B, C`, nên model đúng cả ba. Giả sử trước khi tính accuracy, tức tỉ lệ nhãn dự đoán khớp đáp án, dự đoán bị biến thành một cột `(3, 1)`:

In [ ]:
truth = np.array(["A", "B", "C"])
predicted_column = predicted[:, None]
wrong_comparison = predicted_column == truth
print("shape so sánh sai:", wrong_comparison.shape)
print("accuracy tính nhầm:", round(float(wrong_comparison.mean()), 3))

# Mỗi mẫu có đúng một nhãn, nên đưa dự đoán về lại một trục.
aligned = predicted_column.ravel()
assert aligned.shape == truth.shape
print("accuracy theo từng mẫu:", float((aligned == truth).mean()))

shape so sánh sai: (3, 3)
accuracy tính nhầm: 0.333
accuracy theo từng mẫu: 1.0


Phép so sánh sai tạo ra một bảng 3 × 3: nhãn của mỗi mẫu bị đem so với cả ba đáp án, chỉ ba ô trên đường chéo là khớp, nên ra `3/9 = 0.333`. So đúng từng cặp thì được 1.0. Chương trình không báo lỗi gì ở phép broadcast sai, nên thói quen in shape trước khi tính metric quan trọng hơn ta tưởng.